In [23]:
import pandas as pd

In [28]:
df = pd.read_csv('../../data/processed/ktc_value_histories_20241229-122823.csv')
df.slug.nunique()

500

In [29]:
# largest spike
df = df.sort_values(by=['slug', 'date'])
n_steps = 7
delta_col = f'delta_{n_steps}'
df[delta_col] = df.groupby('slug')['value'].diff(n_steps)
print(df[delta_col].max())
df.loc[df[delta_col].idxmax()]

2483.0


date                                210916
value                                 3921
slug                  elijah-mitchell-1013
player_name                Elijah Mitchell
position                                RB
team                                   SFO
age                                   26.7
birthday                       894085200.0
height_feet                              5
height_inches                           10
weight                                 200
draft_year                            2021
seasons_experience                       3
pick_round                               6
pick_num                                10
delta_7                             2483.0
Name: 320922, dtype: object

In [27]:
def find_top_n_spikes(df, value_col='value', group_col='slug', n_steps=7, top_n=5):
   # Sort and calculate deltas
   df = df.sort_values(by=[group_col, 'date'])
   delta_col = f'delta_{n_steps}'
   df[delta_col] = df.groupby(group_col)[value_col].diff(n_steps)
   
   # Get top n spikes
   top_spikes = df.nlargest(top_n, delta_col)
   
   # Add the starting values and dates for each spike
   results = []
   for _, row in top_spikes.iterrows():
       start_idx = row.name - n_steps if row.name - n_steps >= 0 else 0
       results.append({
           'slug': row[group_col],
           'spike': row[delta_col],
           'start_date': df.loc[start_idx, 'date'],
           'end_date': row['date'],
           'start_value': df.loc[start_idx, value_col],
           'end_value': row[value_col],
           'percent_change': ((row[value_col] - df.loc[start_idx, value_col]) / 
                            df.loc[start_idx, value_col] * 100)
       })
   
   return pd.DataFrame(results)

find_top_n_spikes(df, 'value', 'slug', 7, 5)

/tmp/ipykernel_19729/2000160455.py:21: RuntimeWarning: divide by zero encountered in scalar divide
  'percent_change': ((row[value_col] - df.loc[start_idx, value_col]) /


,slug,spike,start_date,end_date,start_value,end_value,percent_change
0,elijah-mitchell-1013,2483.0,2021-09-09,2021-09-16,1438,3921,172.670376
1,myles-gaskin-3,2465.0,2020-10-01,2020-10-08,765,3230,322.222222
2,myles-gaskin-3,2403.0,2020-09-30,2020-10-07,765,3168,314.117647
3,elijah-mitchell-1013,2378.0,2021-09-12,2021-09-19,1459,3837,162.988348
4,isaiah-hodgins-561,2320.0,2022-12-29,2023-01-05,0,2320,inf
